---
title: "Building TCP on UDP"
version: 2
abstract: >
    FIXME
syllabus:
-   Explain how sequence numbers, acknowledgments, and retransmission together guarantee reliable delivery over an unreliable network.
-   Describe how a sliding window allows multiple segments to be in flight simultaneously and why this improves throughput.
-   Explain the AIMD (additive increase, multiplicative decrease) algorithm and why it causes multiple flows sharing a link to converge to fair shares.
-   Distinguish between the congestion window and the receive window, and explain what each one limits.
---

**In Development**

When you open a website, stream a video, or send an email,
you're almost certainly using [%g tcp "Transmission Control Protocol" %] (TCP).
TCP provides reliable, ordered delivery of data over the internet,
handling packet loss, reordering, and congestion automatically.
But TCP itself runs on top of UDP,
which only provides best-effort, unordered packet delivery with no guarantees.

Building a simplified TCP on top of UDP
demonstrates the core mechanisms that make reliable communication possible over unreliable networks:

-   Sequence numbers:
    Each byte has a sequence number.
    Receivers use these to detect missing data and reorder packets.

-   Acknowledgments (ACKs):
    Receivers send ACKs indicating what data they've received.
    Senders use ACKs to know what to retransmit.

-   Retransmission:
    If an ACK doesn't arrive within a timeout, the sender retransmits the data.

-   Cumulative ACKs:
    ACKs indicate that the receiver has received everything up to a particular message,
    which simplifies acknowledgment logic.

-   Sliding windows:
    Sender can have multiple unacknowledged packets in flight
    to maintain throughput despite round-trip latency.

Our TCP-over-UDP implementation consists of:

1.  `tcp_types.py`: core data structures.
1.  `unreliable_network.py`: simulates packet loss, reordering, duplication.
1.  `tcp_connection.py`: TCP connection with reliability mechanisms.
1.  `tcp_applications.py`: client and server applications

Let's examine each component in turn.

## Data Structures {: #tcp-data}

We start by defining the six types of packets our protocol will use:

[%inc tcp_types.py mark=packettypes %]

and the seven different states that a connection can be in:

[%inc tcp_types.py mark=connectionstate %]

The `Packet` structure mirrors a real TCP/IP packet header with source and destination addressing,
sequence and acknowledgment numbers,
packet type,
and payload data:

[%inc tcp_types.py mark=packet %]

## Unreliable Network Layer {: #tcp-network}

Before building TCP, we need to simulate UDP's unreliable delivery.
`UnreliableNetwork` tracks per-packet statistics
and maintains a registry mapping address:port pairs to their receive queues:

[%inc unreliable_network.py mark=network_init %]

`send_packet` applies the configured loss and duplication rates
before calling `_deliver_packet`,
which adds a random delay and optionally increases it to simulate reordering:

[%inc unreliable_network.py mark=network_send %]

This simulates the way packets in real networks can be lost, delayed, reordered, or duplicated.
The network maintains a registry of endpoints and routes packets accordingly.

## TCP Connection {: #tcp-connection}

The TCP connection implements reliability over the unreliable network:

[%inc tcp_connection.py mark=tcpinit %]

The connection maintains send and receive buffers.
The send buffer holds unacknowledged segments for potential retransmission.
The receive buffer handles out-of-order delivery by holding segments until gaps are filled.

The `run` loop reads packets from the receive queue and dispatches each one through `handle_packet`,
which routes SYN, SYN-ACK, ACK, and DATA packets to the appropriate handler:

[%inc tcp_connection.py mark=tcp_run %]

## Three-Way Handshake {: #tcp-handshake}

TCP connection establishment uses three packets to synchronize state:

[%inc tcp_connection.py mark=tcpconnect %]

The handshake sequence is:

1.  Client → Server: SYN (seq=x)
1.  Server → Client: SYN-ACK (seq=y, ack=x+1)
1.  Client → Server: ACK (ack=y+1)

This synchronizes sequence numbers and establishes bidirectional communication.

The server side of the handshake is handled by `listen_and_accept`,
which waits for a SYN packet and then delegates the response to `handle_syn`:

[%inc tcp_connection.py mark=server_accept %]

## Sliding Window Protocol {: #tcp-window}

The sliding window allows multiple packets in flight:

[%inc tcp_connection.py mark=tcpsend %]

The sender can have `window_size` unacknowledged packets in flight.
This maintains throughput even with high latency:
new packets are being sent while waiting for ACKs from earlier packets.

## Retransmission on Timeout {: #tcp-retrans}

If an ACK doesn't arrive within the timeout period,
the segment is retransmitted.
We use a separate `Process` to simulate each retransmission timer:

[%inc tcp_connection.py mark=retransmission %]

Each time we sent a segment, we create a `RetransmissionTimer`:

```python

# In TCPConnection.send():
await self.network.send_packet(segment)

# Add to send buffer
self.send_buffer.append(buffer_entry)

# Start retransmission timer
RetransmissionTimer(self._env, self, buffer_entry)
```

Each segment has its own timer `Process`.
If the segment hasn't been acknowledged (i.e., is still in the send buffer)
when the timer expires,
it is retransmitted and a new timer starts.

When a DATA packet arrives,
`handle_data` adds it to the receive buffer,
extracts any newly contiguous data to deliver to the application,
and sends a cumulative ACK:

[%inc tcp_connection.py mark=handle_data %]

## Handling Out-of-Order Delivery {: #tcp-order}

The receive buffer handles segments arriving out of order:

[%inc tcp_types.py mark=receivebuffer %]

Segments are held until all gaps are filled.
When a contiguous block of data is available,
it's delivered to the application in order.

## Cumulative Acknowledgments {: #tcp-ack}

TCP uses cumulative ACKs,
i.e.,
each ACK indicates all data up to a sequence number has been received:

[%inc tcp_connection.py mark=handleack %]

A single ACK can acknowledge multiple segments.
This is simpler than selective acknowledgments and works well when most data arrives in order.

## Basic Example {: #tcp-basic}

Let's see TCP in action:

[%inc ex_basic.py mark=basicexample %]
[%inc ex_basic.out %]

Despite 15% packet loss and reordering,
TCP successfully delivers the complete message in order.

## High Loss Scenario {: #tcp-highloss}

Let's test TCP under extreme conditions:

[%inc ex_loss_recovery.py mark=highlossexample %]
[%inc ex_loss_recovery.out %]

Even with 40% loss, TCP delivers the complete message:
there are many retransmissions but eventual success.

## Key TCP Concepts

**Sequence Number Space**:
Each byte transmitted has a unique sequence number.
For a message "Hello":

- Byte 'H' might be seq=1000
- Byte 'e' is seq=1001
- Byte 'l' is seq=1002
- And so on...

When acknowledging, the receiver sends back seq=1005,
meaning, "I've received everything up to 1005."

**Window Size and Throughput**:
The window size determines maximum throughput.
With:

- Window size = 4 segments
- Segment size = 1000 bytes
- Round-trip time = 0.5 seconds

the maximum throughput is approximately (4 × 1000) / 0.5 = 8000 bytes/second.
Larger windows enable higher throughput but require more buffering.

**Adaptive Retransmission**:
Our implementation uses a fixed timeout.
Real TCP measures round-trip time and adapts the timeout:

```
RTO = smoothed_RTT + 4 × RTT_variance
```

This balances responsiveness (short timeout) with avoiding spurious retransmissions (long timeout).

**Fast Retransmit**:
An optimization not in our implementation
is that if the sender receives three duplicate ACKs for the same sequence number,
it immediately retransmits without waiting for timeout.
This recovers from loss faster.

## Congestion Control {: #tcp-congestion}

Our implementation uses a fixed window size.
Real TCP adapts its window dynamically based on network conditions
using [%g aimd "Additive Increase, Multiplicative Decrease" %] (AIMD).

The sender maintains a *congestion window* (`cwnd`).
It has two phases:

**Slow Start**: begin with `cwnd = 1` segment and double it every RTT.
Double may sound fast, but starting from 1 prevents overwhelming a slow link on connection open.

**Congestion Avoidance**: once `cwnd` exceeds a threshold (`ssthresh`),
increase by 1 segment per RTT instead of doubling.
This probes for additional capacity without overshooting.

**Multiplicative Decrease**: when a segment times out, halve `cwnd` and `ssthresh`.
Starting slow again recovers quickly because each RTT doubles `cwnd` back to `ssthresh`.

**Fast Retransmit**: three consecutive duplicate ACKs for the same sequence number
means a segment was probably lost (not just delayed).
The sender retransmits immediately without waiting for the timeout—
this recovers from a single drop faster than a 2-second timeout would.
On a fast retransmit, `cwnd` is halved (less severe than a timeout, which resets to 1).

Here is the congestion state machine:

[%inc congestion_control.py mark=cwnd_state %]

And the sender that uses it:

[%inc congestion_control.py mark=cc_sender %]

AIMD is the reason the internet does not collapse under load.
If all senders backed off linearly on loss, they would overshoot and undershoot repeatedly.
The multiplicative decrease is aggressive enough to clear congestion quickly,
while additive increase is conservative enough that multiple flows converge to equal shares.

## Real-World Considerations

Production TCP implementations include features we've omitted:

-   **Selective Acknowledgments (SACK)** acknowledge non-contiguous blocks of data,
    which allows efficient retransmission of only the missing segments.
    Without SACK, a lost segment in the middle of a window forces retransmission of everything after it.

-   **Flow Control** (distinct from congestion control):
    the *receiver* advertises its available buffer space in each ACK's window field.
    The sender limits in-flight data to the min of `cwnd` and the receiver's window.
    This prevents a fast sender from overwhelming a slow receiver's buffer.
    Our implementation uses `cwnd` as the only window—adding the receiver window is left as an exercise.

-   **Nagle's Algorithm** batches small writes to reduce overhead.

-   **Path MTU Discovery**
    determines maximum packet size without fragmentation by testing increasingly large packets.

-   **Timestamp Options**
    improve RTT estimation and detect wrapped sequence numbers on high-speed networks.

## Performance Analysis

Let's analyze our TCP implementation's efficiency:
With 15% loss and 4-segment window:

-   approximately 15% of packets need retransmission;
-   the window allows 4 segments in flight; and
-   the average delay per segment is approximately RTT + (loss_rate × timeout)

The throughput efficiency is therefore

```
efficiency = successful_transmission_rate / available_bandwidth
           ≈ (1 - loss_rate) / (1 + loss_rate × retransmissions)
```

<section class="exercises" markdown="1">

## Exercises {: #tcp-exercises}

1.  Run the basic example with 0% loss, then with 30% loss.
    Count the number of retransmissions in each case.
    Now increase the window size from 4 to 8.
    Does a larger window help or hurt under high loss?
    Why?

2.  The fixed retransmission timeout is set to a constant.
    Real TCP uses the Karn/Jacobson formula:
    `RTO = smoothed_RTT + 4 * RTT_variance`.
    Add `smoothed_rtt` and `rtt_variance` fields to `TCPConnection`.
    When an ACK arrives, update them:
    `smoothed_rtt = 0.875 * smoothed_rtt + 0.125 * sample_rtt`
    and `rtt_variance = 0.75 * rtt_variance + 0.25 * |sample_rtt - smoothed_rtt|`.
    (Starter: record `sent_time` in `SegmentBuffer` and compute `sample_rtt` when the ACK arrives.)

3.  Trace through the congestion control state machine for this sequence:
    - cwnd starts at 1, ssthresh = 8.
    - ACKs arrive for seq 0, 1, 2, 3 (no loss).
    - What is cwnd after each ACK (draw the table)?
    - Segment 4 times out.
    - What are the new cwnd and ssthresh?
    - ACKs arrive for seq 4, 5, 6.
    - What is cwnd after each ACK?

4.  The receiver advertises a window in every ACK (flow control).
    Suppose the receiver's buffer is only 2 segments.
    Modify `CongestionControlledSender` to accept a `receiver_window` parameter
    and limit in-flight data to `min(cwnd, receiver_window)`.
    What happens when the receiver's window is smaller than cwnd?

5.  SACK (Selective Acknowledgment) allows the receiver to acknowledge
    non-contiguous ranges of received data.
    Without SACK, if segment 3 is lost but segments 4–7 arrive,
    the sender must retransmit segment 3 and cannot know whether 4–7 need retransmission.
    Describe the data structure the receiver would need to send SACK information,
    and how the sender would use it to retransmit only the missing segment.

</section>